In [ ]:
# [CELL 1] INSTALL DEPENDENCIES & DOWNLOAD WEIGHTS
!pip install -q tiktoken datasets huggingface_hub

import os
import torch
import numpy as np
import tiktoken
from datasets import load_dataset
from huggingface_hub import hf_hub_download

# 1. Download Model Weights from your Hugging Face repo
# (You will update 'TheToyChest/DPM-15L' to your actual repo name)
REPO_ID = "TheToyChest/DPM-15L"
FILENAME = "SEED 999 DPM 15L 25Million.pt"

print(f"Downloading DPM weights from {REPO_ID\}...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
print(f"Weights downloaded to: {model_path\}")

# 2. Automatically build the Wikitext-103 validation binary
print("Downloading and tokenizing Wikitext-103 validation set...")
enc = tiktoken.get_encoding("gpt2")
wiki_dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="validation")

# Tokenize and save to a local binary file for memory mapping
val_ids = []
for text in wiki_dataset['text']:
    if text.strip():
        val_ids.extend(enc.encode_ordinary(text))

wiki_val_arr = np.array(val_ids, dtype=np.uint16)
wiki_val_arr.tofile("wiki_val.bin")
print(f"wiki_val.bin created with {len(wiki_val_arr):,\} tokens.")$0

In [ ]:
# [CELL 2] ARCHITECTURE & CONFIGURATION
import torch.nn as nn
import torch.nn.functional as F
import math

class Config:
    attn_mode = 'dpa'
    dpaloss = True
    trn_shft_loss = False
    use_rope = True

    seq_len = 512
    embed_dim = 256
    num_layers = 15
    num_heads = 4
    ffn_dim = 1024
    dropout = 0.0

    preNorm = False
    postNorm = True
    use_qk_norm = True
    vocab_size = 50257

    logitsclamp = 50.0
    att_shft_pos = False
    clamp_shft = False

    # Set this to match the ablation state of the weights being tested
    continuous_abla = False

class UniversalAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.num_heads = cfg.num_heads
        self.head_dim = cfg.embed_dim // cfg.num_heads

        self.W_q = nn.Linear(cfg.embed_dim, cfg.embed_dim, bias=False)
        self.W_k = nn.Linear(cfg.embed_dim, cfg.embed_dim, bias=False)
        self.W_v = nn.Linear(cfg.embed_dim, cfg.embed_dim, bias=False)
        self.W_o = nn.Linear(cfg.embed_dim, cfg.embed_dim, bias=False)

        if getattr(cfg, 'att_shft_pos', False):
            self.raw_score_bias = nn.Parameter(torch.tensor(0.0))
        else:
            self.score_bias = nn.Parameter(torch.tensor(0.5))

        if cfg.use_rope:
            inv_freq = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
            t = torch.arange(cfg.seq_len)
            freqs = torch.outer(t, inv_freq)
            emb = torch.repeat_interleave(freqs, 2, dim=-1)
            self.register_buffer('rope_cos', emb.cos()[None, None, :, :])
            self.register_buffer('rope_sin', emb.sin()[None, None, :, :])

        if getattr(cfg, 'use_qk_norm', False):
            self.q_norm = nn.RMSNorm(self.head_dim)
            self.k_norm = nn.RMSNorm(self.head_dim)

        if cfg.attn_mode == 'dpa':
            self.poly_a = nn.Parameter(torch.tensor(1.0))
            self.poly_b = nn.Parameter(torch.tensor(1.0))
            self.poly_z_exp = nn.Parameter(torch.tensor(1.5))
            c_mask = torch.tril(torch.ones(cfg.seq_len, cfg.seq_len)).view(1, 1, cfg.seq_len, cfg.seq_len)
            self.register_buffer('causal_mask', c_mask)

    def forward(self, x):
        B, T, C = x.size()
        Q = self.W_q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        if getattr(self.cfg, 'use_qk_norm', False):
            Q = self.q_norm(Q)
            K = self.k_norm(K)

        if self.cfg.use_rope:
            cos = self.rope_cos[:, :, :T, :]
            sin = self.rope_sin[:, :, :T, :]
            def rotate_interleaved(tensor):
                x1 = tensor[..., 0::2]
                x2 = tensor[..., 1::2]
                return torch.stack((-x2, x1), dim=-1).reshape(tensor.shape)
            Q = (Q * cos) + (rotate_interleaved(Q) * sin)
            K = (K * cos) + (rotate_interleaved(K) * sin)

        if self.cfg.attn_mode == 'dpa':
            causal_mask = self.causal_mask[:, :, :T, :T]
            thermal_div = max(8.0, math.sqrt(self.head_dim))

            scores = torch.matmul(Q, K.transpose(-2, -1)) / thermal_div
            scores = scores.masked_fill(causal_mask == 0, -1e4)

            if Config.att_shft_pos:
                score_bias = F.softplus(self.raw_score_bias)
            else:
                score_bias = self.score_bias.clamp(min=0.0001) if Config.clamp_shft else self.score_bias

            scores = scores + score_bias
            scores = scores.masked_fill(causal_mask == 0, 0.0)

            r_attn = F.relu(scores)
            r_safe = r_attn + 1e-6

            a = F.softplus(self.poly_a)
            b = F.softplus(self.poly_b)
            z_exp = 2.0 + F.softplus(self.poly_z_exp)

            attn_raw = (a * (r_safe ** 2.0)) + (b * (r_safe ** z_exp))
            attn_sum = attn_raw.sum(dim=-1, keepdim=True).clamp(min=1e-6)
            attn = attn_raw / attn_sum
            attn_out = torch.matmul(attn, V)
        else:
            attn_out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(attn_out)

class StandardTransformerBlock(nn.Module):
    def __init__(self, cfg, layer_idx=0):
        super().__init__()
        self.cfg = cfg
        self.ln1 = nn.RMSNorm(cfg.embed_dim)
        self.ln2 = nn.RMSNorm(cfg.embed_dim)

        if getattr(cfg, 'postNorm', False):
            self.ln1_post = nn.RMSNorm(cfg.embed_dim)
            self.ln2_post = nn.RMSNorm(cfg.embed_dim)

        self.attn = UniversalAttention(cfg)
        self.ffn = nn.Sequential(
            nn.Linear(cfg.embed_dim, cfg.ffn_dim),
            nn.GELU(),
            nn.Linear(cfg.ffn_dim, cfg.embed_dim),
        )

    def forward(self, x):
        norm_x = self.ln1(x) if getattr(self.cfg, 'preNorm', True) else x
        attn_out = self.attn(norm_x)
        if getattr(self.cfg, 'postNorm', False):
            attn_out = self.ln1_post(attn_out)
        x = x + attn_out

        norm_x2 = self.ln2(x) if getattr(self.cfg, 'preNorm', True) else x
        ffn_out = self.ffn(norm_x2)
        if getattr(self.cfg, 'postNorm', False):
            ffn_out = self.ln2_post(ffn_out)
        x = x + ffn_out
        return x

class StandardTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.token_embed = nn.Embedding(cfg.vocab_size, cfg.embed_dim)

        if not cfg.use_rope:
            self.pos_embed = nn.Embedding(cfg.seq_len, cfg.embed_dim)

        self.layers = nn.ModuleList([StandardTransformerBlock(cfg, layer_idx=i) for i in range(cfg.num_layers)])
        self.ln_f = nn.RMSNorm(cfg.embed_dim)

        self.vocab_proj = nn.Linear(cfg.embed_dim, cfg.vocab_size, bias=False)
        self.vocab_proj.weight = self.token_embed.weight

        self.loss_poly_a = nn.Parameter(torch.tensor(1.0))
        self.loss_poly_b = nn.Parameter(torch.tensor(1.0))
        self.loss_poly_z_exp = nn.Parameter(torch.tensor(1.5))
        self.loss_bias = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        B, T = x.size()
        embed = self.token_embed(x)
        if not self.cfg.use_rope:
            positions = torch.arange(0, T, device=x.device).unsqueeze(0).expand(B, T)
            embed = embed + self.pos_embed(positions)

        x = embed
        for layer in self.layers:
            x = layer(x)

        x = self.ln_f(x)
        return self.vocab_proj(x), x$0

In [ ]:
# [CELL 3] EVALUATION LOGIC & EXECUTION
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def evaluate_on_wikitext(model_path, num_eval_steps=1000):
    print(f"\\n{'='*60\}")
    eval_type = "DPM / NO-SOFTMAX" if Config.dpaloss else "STANDARD SOFTMAX"
    print(f" ZERO-SHOT ROBUSTNESS TEST: WIKITEXT-103 ({eval_type\})")
    print(f"{'='*60\}")

    wiki_data = np.memmap("wiki_val.bin", dtype=np.uint16, mode='r')
    print(f"[DATA] Loaded Wikitext-103 Validation Tokens: {len(wiki_data):,\}")

    model = StandardTransformer(Config).to(device)
    print(f"[SYSTEM] Loading trained weights...")
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)

    # Strip compilation artifacts if present
    state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
    uncompiled_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()\}
    model.load_state_dict(uncompiled_state_dict)
    model.eval()

    def get_wiki_batch():
        batch_size = 16
        ix = np.random.randint(0, len(wiki_data) - Config.seq_len - 1, size=(batch_size,))
        offsets = np.arange(Config.seq_len)
        idx = ix[:, None] + offsets
        x = torch.from_numpy(wiki_data[idx].astype(np.int64)).to(device)
        y = torch.from_numpy(wiki_data[idx + 1].astype(np.int64)).to(device)
        return x, y

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    print(f"\\n[SYSTEM] Running Strict Exact-Match (SEM) Evaluation over {num_eval_steps\} batches...")

    with torch.no_grad():
        for step in range(num_eval_steps):
            X_val, Y_val = get_wiki_batch()

            with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits, _ = model(X_val)

                if Config.dpaloss:
                    logits_f32 = logits.float().clamp(min=-Config.logitsclamp, max=Config.logitsclamp)
                    if Config.trn_shft_loss and hasattr(model, 'loss_bias'):
                        logits_f32 = logits_f32 + model.loss_bias

                    # Continuous Ablation Toggle
                    if getattr(Config, 'continuous_abla', False):
                        r_safe = F.elu(logits_f32) + 1.0
                    else:
                        r = F.relu(logits_f32)
                        r_safe = r + 1e-6

                    a = F.softplus(model.loss_poly_a)
                    b = F.softplus(model.loss_poly_b)
                    z_exp = 2.0 + F.softplus(model.loss_poly_z_exp)

                    activations = (a * (r_safe ** 2.0)) + (b * (r_safe ** z_exp))
                    prob_sum = activations.sum(dim=-1, keepdim=True).clamp(min=1e-8)
                    probs = activations / prob_sum

                    log_probs = torch.log(probs.clamp(min=1e-9))
                    batch_nll = F.nll_loss(log_probs.view(-1, Config.vocab_size), Y_val.view(-1))
                    top1_preds = torch.argmax(activations, dim=-1)

                else:
                    logits_f32 = logits.float()
                    batch_nll = F.cross_entropy(logits_f32.view(-1, Config.vocab_size), Y_val.view(-1))
                    top1_preds = torch.argmax(logits_f32, dim=-1)

            total_loss += batch_nll.item()
            total_correct += (top1_preds == Y_val).sum().item()
            total_tokens += Y_val.numel()

            if (step + 1) % 100 == 0:
                current_acc = (total_correct / total_tokens) * 100
                print(f"  Eval Step {step+1:4d\}/{num_eval_steps\} | Running SEM Accuracy: {current_acc:.2f\}%")

    avg_nll = total_loss / num_eval_steps
    final_accuracy = (total_correct / total_tokens) * 100.0
    perplexity = math.exp(avg_nll) if avg_nll < 20 else float('inf')

    print(f"\\n{'='*60\}")
    print(f" WIKITEXT-103 ZERO-SHOT RESULTS ({eval_type\})")
    print(f"{'='*60\}")
    print(f"  NLL Loss:       {avg_nll:.4f\}")
    print(f"  Perplexity:     {perplexity:.2f\}")
    print(f"  SEM Accuracy:   {final_accuracy:.2f\}%")
    print(f"{'='*60\}\\n")

# Run the evaluation using the downloaded weights
evaluate_on_wikitext(model_path, num_eval_steps=1000)$0